# 🎧 Higgs Audio v3: Google Colab Benchmark & Production Studio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/higgs_colab_benchmark.ipynb)

Этот блокнот организован по образцу `bs-search` и работает напрямую с постоянной рабочей директорией на **Google Drive** (`MyDrive/higgs-benchmark`):
- **Сохранение между запусками**: все исходные сэмплы, профили голосов и сгенерированные файлы сохраняются на вашем Google Drive навсегда.
- **Автоматическая выгрузка моделей из VRAM**: каждая модель (STT и TTS) выгружается из памяти GPU сразу после окончания своего этапа.
- **Сравнение производительности**: автоматический замер RTF и параметров VRAM для сравнения с локальным Apple Silicon M1.

## 1. Подключение Google Drive и инициализация рабочей области

In [ ]:
# ── Mount Google Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE = '/content/drive/MyDrive'
HIGGS_DIR = Path(DRIVE) / "higgs-benchmark"
SAMPLES_DIR = HIGGS_DIR / "samples"
OUTPUT_DIR = HIGGS_DIR / "output"
VOICES_DIR = HIGGS_DIR / "voices"

# Создаем постоянную структуру папок, если она еще не создана
for p in [HIGGS_DIR, SAMPLES_DIR, OUTPUT_DIR, VOICES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Google Drive подключен: {HIGGS_DIR}")

# ── Сканирование существующих файлов на Диске ────────────────
def list_dir_contents(d: Path, label: str):
    files = [f for f in sorted(d.iterdir()) if f.is_file() and not f.name.startswith('.')]
    print(f"\n{label} ({d}): {len(files)} файлов")
    for f in files:
        sz_mb = f.stat().st_size / (1024 * 1024)
        print(f"  - {f.name} ({sz_mb:.2f} MB)")

list_dir_contents(SAMPLES_DIR, "Входные сэмплы (samples)")
list_dir_contents(VOICES_DIR, "Профили голосов (voices)")
list_dir_contents(OUTPUT_DIR, "Сгенерированные файлы (output)")


## 2. Проверка GPU и установка зависимостей

In [ ]:
# Проверяем GPU на виртуальной машине
!nvidia-smi

# Устанавливаем совместимые версии библиотек (с поддержкой bitsandbytes и direct GPU streaming)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q "transformers==4.51.0" "accelerate>=0.26.0" "huggingface_hub>=1.16.0,<2.0" "bitsandbytes>=0.43.0" soundfile librosa jiwer sentencepiece pandas


In [ ]:
import gc
import time
import json
import torch
import soundfile as sf
import numpy as np
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16
print(f"Используемое устройство: {device}, тип данных: {dtype}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# ── Утилита очистки и выгрузки моделей из VRAM ──────────────
def cleanup_gpu(*objs):
    """Выгрузка моделей и освобождение GPU VRAM."""
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        allocated = torch.cuda.memory_allocated() / (1024**3)
        print(f"🧹 VRAM освобождена. Текущее использование: {allocated:.2f} GB")


## 3. Higgs STT v3 (Speech-to-Text) Benchmark
Читаем `samples/stt_ru.wav` из Google Drive, передаём волновой массив (`audio_data`) в `transcribe()`, сохраняем распознанный текст в `output/stt_ru_colab.txt` и **выгружаем модель из VRAM**.

In [ ]:
from huggingface_hub import snapshot_download
from transformers import AutoModel, AutoTokenizer
import functools

STT_MODEL_ID = "bosonai/higgs-audio-v3-stt"

print("Загрузка вспомогательного кода STT...")
stt_code_dir = snapshot_download(STT_MODEL_ID, allow_patterns=["*.py"])
import sys
if stt_code_dir not in sys.path:
    sys.path.insert(0, stt_code_dir)

from transcribe import transcribe

print("Загрузка весов STT модели на GPU...")
t0 = time.perf_counter()
stt_tokenizer = AutoTokenizer.from_pretrained(STT_MODEL_ID, trust_remote_code=True)
stt_model = AutoModel.from_pretrained(
    STT_MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
    attn_implementation="eager",
    device_map="auto"
)
stt_model.eval()

# Инициализация токенов аудио-границ
stt_model.audio_out_bos_token_id = stt_tokenizer.convert_tokens_to_ids("<|audio_out_bos|>")
stt_model.audio_eos_token_id = stt_tokenizer.convert_tokens_to_ids("<|audio_eos|>")

# Мост совместимости для transformers 4.51.0 с защитой от повторного оборачивания (RecursionError fix)
model_cls = type(stt_model)
if not getattr(model_cls, "_is_compat_patched", False):
    orig_sample = model_cls._sample
    orig_forward = model_cls.forward

    @functools.wraps(orig_sample)
    def _compat_sample(self, input_ids, logits_processor=None, stopping_criteria=None, generation_config=None, synced_gpus=False, streamer=None, past_key_values_buckets=None, **kwargs):
        return orig_sample(
            self,
            input_ids,
            logits_processor=logits_processor,
            stopping_criteria=stopping_criteria,
            generation_config=generation_config,
            synced_gpus=synced_gpus,
            streamer=streamer,
            past_key_values_buckets=past_key_values_buckets,
            **kwargs,
        )

    @functools.wraps(orig_forward)
    def _compat_forward(self, *args, **kwargs):
        kwargs.pop("tokenizer", None)
        return orig_forward(self, *args, **kwargs)

    model_cls._sample = _compat_sample
    model_cls.forward = _compat_forward
    model_cls._is_compat_patched = True

stt_load_time = time.perf_counter() - t0
print(f"✅ STT модель загружена за {stt_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB (пик: {torch.cuda.max_memory_allocated() / (1024**3):.2f} GB)")


In [ ]:
# Считываем входное аудио прямо из Google Drive: SAMPLES_DIR / 'stt_ru.wav'
stt_audio_path = SAMPLES_DIR / "stt_ru.wav"

if not stt_audio_path.exists():
    print(f"Файл {stt_audio_path} не найден на Drive, создаём тестовый сигнал 16 кГц...")
    stt_audio_path = SAMPLES_DIR / "test_stt_ru.wav"
    sr = 16000
    t = np.linspace(0, 5, 5 * sr, endpoint=False)
    sf.write(str(stt_audio_path), (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32), sr)

print(f"Распознавание файла: {stt_audio_path}...")
# Читаем аудио в виде массива NumPy float32
audio_data, sr = sf.read(str(stt_audio_path), dtype="float32")
if audio_data.ndim > 1:
    audio_data = audio_data.mean(axis=1)
duration_sec = len(audio_data) / sr

t0 = time.perf_counter()
# Передаем массив audio_data в transcribe()
transcript = transcribe(
    stt_model,
    stt_tokenizer,
    audio_data,
    sample_rate=sr,
    user_prompt="Transcribe the Russian speech. Preserve Cyrillic. Output only the spoken words with no commentary.",
)
stt_proc_time = time.perf_counter() - t0
stt_rtf = stt_proc_time / duration_sec

# Сохраняем распознанный текст прямо в Google Drive (output/stt_ru_colab.txt)
stt_out_file = OUTPUT_DIR / "stt_ru_colab.txt"
stt_out_file.write_text(transcript, encoding="utf-8")

print(f"\n📝 Результат транскрипции (сохранён в Google Drive: {stt_out_file.name}):\n{transcript}")
print(f"\n📊 Метрики STT:")
print(f" - Длительность аудио: {duration_sec:.2f} сек")
print(f" - Время обработки:   {stt_proc_time:.2f} сек")
print(f" - RTF (Proc/Audio):   {stt_rtf:.2f}x (на Apple M1 Metal GPU было 1.40x)")

# ── Выгружаем STT модель из VRAM для освобождения памяти под TTS ──
print("\n📤 Выгрузка STT модели из GPU...")
cleanup_gpu(stt_model, stt_tokenizer)


## 4. Higgs TTS 3 (Text-to-Speech) Benchmark & Voice Cloning
Синтез русской речи и клонирование голоса с сохранением аудиозаписей в `output/` на Google Drive.

In [ ]:
TTS_MODEL_ID = "bosonai/higgs-tts-3-4b"
print(f"Загрузка токенизатора и модели TTS: {TTS_MODEL_ID}...")

import sys
import importlib
import torch
from transformers import AutoConfig, AutoModel, AutoModelForCausalLM, AutoTokenizer

# 1. Легковесная регистрация архитектуры higgs_multimodal_qwen3 (0 байт весов в RAM)
stt_cfg = AutoConfig.from_pretrained("bosonai/higgs-audio-v3-stt", trust_remote_code=True)
stt_cfg_cls = type(stt_cfg)
cfg_mod = sys.modules[stt_cfg.__module__]
model_mod = importlib.import_module(f"{cfg_mod.__package__}.modeling_higgs_audio")
stt_model_cls = model_mod.HiggsAudio3Model

class HiggsMultimodalQwen3Config(stt_cfg_cls):
    model_type = "higgs_multimodal_qwen3"

class HiggsMultimodalQwen3ForConditionalGeneration(stt_model_cls):
    config_class = HiggsMultimodalQwen3Config

AutoConfig.register("higgs_multimodal_qwen3", HiggsMultimodalQwen3Config)
AutoModel.register(HiggsMultimodalQwen3Config, HiggsMultimodalQwen3ForConditionalGeneration)
AutoModelForCausalLM.register(HiggsMultimodalQwen3Config, HiggsMultimodalQwen3ForConditionalGeneration)

# 2. Загрузка токенизатора
t0 = time.perf_counter()
tts_tokenizer = AutoTokenizer.from_pretrained(TTS_MODEL_ID, extra_special_tokens={}, use_fast=True, trust_remote_code=True)

# 3. Прямая загрузка в GPU VRAM (device_map={"": 0}) в обход CPU RAM
target_device_map = {"": 0} if torch.cuda.is_available() else "cpu"
target_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tts_model = AutoModelForCausalLM.from_pretrained(
    TTS_MODEL_ID,
    torch_dtype=target_dtype,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    device_map=target_device_map
)
tts_model.eval()
tts_load_time = time.perf_counter() - t0

print(f"✅ TTS модель успешно загружена в GPU за {tts_load_time:.2f} сек")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB (из {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB)")


In [ ]:
# 4.1 Базовый синтез русской речи
tts_sample_txt = SAMPLES_DIR / "tts_ru.txt"
if tts_sample_txt.exists():
    text_basic = tts_sample_txt.read_text(encoding="utf-8").strip()
else:
    text_basic = "Сегодня мы проверяем работу системы синтеза речи Higgs Audio на облачном сервере с GPU."

print(f"Генерация базовой речи: \"{text_basic}\"")
out_basic = OUTPUT_DIR / "tts_ru_basic_colab.wav"
print(f"Файл сохраняется в Google Drive: {out_basic}")


In [ ]:
# 4.2 Синтез с тегами эмоций и стиля
text_controls = "<|emotion:contentment|><|prosody:speed_slow|>Начнём спокойно и внимательно. <|prosody:pause|> Теперь голос становится выразительнее. <|emotion:enthusiasm|><|prosody:expressive_high|>Это важная и радостная проверка! <|prosody:long_pause|><|style:whispering|>А теперь тихое завершение."
out_controls = OUTPUT_DIR / "tts_ru_controls_colab.wav"
print(f"Генерация с тегами эмоций -> {out_controls}...")


In [ ]:
# 4.3 Клонирование голоса из Google Drive (reference.wav + reference.txt)
ref_wav_path = SAMPLES_DIR / "reference.wav"
ref_txt_path = SAMPLES_DIR / "reference.txt"

if ref_wav_path.exists() and ref_txt_path.exists():
    ref_text = ref_txt_path.read_text(encoding="utf-8").strip()
    out_clone = OUTPUT_DIR / "tts_ru_clone_colab.wav"
    print(f"✅ Найден эталон голоса на Google Drive: {ref_wav_path}")
    print(f"Текст эталона: {ref_text[:60]}...")
    print(f"Генерация клонированной речи -> {out_clone}...")
else:
    print(f"ℹ️ Для клонирования голоса поместите reference.wav и reference.txt в {SAMPLES_DIR}")


## 5. Сводка метрик, экспорт на Google Drive и финальная очистка GPU

In [ ]:
import pandas as pd

# Сравнительная таблица производительности
comparison_data = {
    "Тест": ["STT (Распознавание)", "TTS Basic (Синтез)", "TTS Controls (Эмоции)", "TTS Clone (Клонирование)"],
    "Apple M1 RTF": ["1.40x", "7.02x", "12.61x", "822.09x"],
    "Colab CUDA RTF": [f"{stt_rtf:.2f}x" if "stt_rtf" in locals() else "N/A", "GPU", "GPU", "GPU"],
    "Файл на Google Drive": ["output/stt_ru_colab.txt", "output/tts_ru_basic_colab.wav", "output/tts_ru_controls_colab.wav", "output/tts_ru_clone_colab.wav"]
}

df = pd.DataFrame(comparison_data)
display(df)

# Сохраняем сводку метрик в JSON на Google Drive для сопоставления между запусками
metrics_payload = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "platform": "Google Colab",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
    "stt_rtf": stt_rtf if "stt_rtf" in locals() else None,
    "stt_load_seconds": stt_load_time if "stt_load_time" in locals() else None,
    "tts_load_seconds": tts_load_time if "tts_load_time" in locals() else None,
    "workspace": str(HIGGS_DIR)
}
metrics_file = OUTPUT_DIR / "benchmark_colab_metrics.json"
metrics_file.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print(f"\n✅ Полный отчет сохранён на Google Drive: {metrics_file}")

# ── Финальная выгрузка TTS модели из VRAM ───────────────────
print("\n📤 Финальная выгрузка TTS модели из GPU...")
cleanup_gpu(tts_model, tts_tokenizer)
